# Lesson 29 — Reformulations and Symmetry

## Learning objectives

1. Recognize symmetric MILPs and the cost they impose on B&B.
2. Apply orbital, lex, and isomorphism-pruning techniques.
3. Recognize convexification reformulations: extended formulations, perspective.
4. Know which reformulations `discopt` applies for you, which it leaves to you, and how to get suggestions (`analyze_reformulations`, the `/reformulate` skill).

## 1. Symmetry hurts B&B

If two variables $x_i, x_j$ are interchangeable (the constraint matrix is invariant under their swap), B&B explores symmetric subtrees independently. Each leaves equivalent solutions multiple times in the tree {cite:p}`Margot2010`.

Examples: bin packing with identical bins; multi-machine scheduling with identical machines; coloring with interchangeable colors.

## 2. Symmetry breaking

- **Lex constraints:** $x_1 \preceq_{\text{lex}} x_2 \preceq \dots$ — valid **only when the variables lie in a single orbit of the formulation's symmetry group**, i.e., they really are interchangeable. Adding lex on non-symmetric variables can cut off optimal solutions.
- **Orbital fixing/branching** {cite:p}`Margot2010`: at each node, identify orbits of the symmetry group; fix all members of an orbit consistently.
- **Isomorphism pruning:** in heavily symmetric problems (e.g., graph coloring), prune nodes that are isomorphic to one already explored.

Modern solvers detect simple symmetries automatically; complex ones require user input.

## 3. Reformulation as a reasoning task

Three families of reformulations matter most:

- **Extended formulations:** add auxiliary variables to make the formulation tighter (smaller LP-relaxation gap). Classic example: SOS2 piecewise via $\lambda$-binaries vs SOS-native.
- **Perspective reformulation:** for an *indicator-conditional* convex epigraph constraint of the form "$y = 1 \Rightarrow f(x) \le t$ and $y = 0 \Rightarrow x = 0$" with $y \in \{0, 1\}$ and $f$ convex, the convex hull of the disjunction is the **perspective** $y f(x / y) \le t$ (with the convention $0 \cdot f(0/0) = 0$). This is strictly tighter than the corresponding big-M and is a workhorse for fixed-cost / indicator constraints in MINLP {cite:p}`Trespalacios2015,Grossmann2013`.
- **Convexification of bilinear terms:** McCormick (Lesson 16) is one option; if the bilinear term has special structure there may be a stronger formulation. When $x_i, x_j$ are both binary, $w_{ij} = x_i x_j$ is linearized *exactly* by $w_{ij} \le x_i$, $w_{ij} \le x_j$, $w_{ij} \ge x_i + x_j - 1$ (Fortet/Glover), turning a binary quadratic program into a MILP.

{cite:p}`Liberti2009` is a systematic catalogue.

## 4. What `discopt` does for you — and what it doesn't

There is no single "reformulate" pass to call. Some reformulations happen
automatically inside `solve()`; others are your job.

**Automatic:**

- **Bound tightening** — root FBBT/OBBT presolve and in-tree FBBT
  (Lessons 15, 21) tighten the boxes every envelope and big-M depends on.
- **Big-M from bounds** — `m.if_then`, `m.either_or` and
  `m.complementarity` derive each $M$ from interval/LP bounds on the row
  (refusing when no finite $M$ exists), and `gdp_method="hull"` gives the
  convex-hull reformulation (Lesson 23).
- **Relaxations of nonconvex terms** — factorable lifting with McCormick,
  RLT, PSD/SOC cuts and αBB (Lesson 16; tune with `partitions=`, `rlt=`).
- **Convexity detection** — a model proved convex skips spatial branching.
- **Binary products** — a model whose only nonlinearity is products of
  binaries (a quadratic knapsack `x @ Q @ x`, a `x[i] * x[j]` row) gets the
  exact Fortet/Glover linearization of §3 and goes to the MILP route; check
  `r.algorithm_route`.
- **MILP symmetry** — pure MILPs are answered by HiGHS by default, which
  runs its own symmetry detection. That is why the cell below shows one
  node with or without lex constraints on that route;
  `m.solve(milp_backend="native")` runs `discopt`'s own MILP tree instead.

**Not automatic:** `discopt`'s own branch-and-bound does no orbital fixing
or orbital branching, and it never adds symmetry-breaking constraints. (The
Rust presolve contains an experimental pass that *detects* candidate
variable orbits in polynomial models, but nothing acts on its output.) Lex
constraints, perspective reformulations and extended formulations are
modelling decisions you make, and you have to verify they are valid.

**Suggestions, not transformations:**
`discopt.llm.reformulation.analyze_reformulations(m)` runs deterministic
checks for oversized big-M, weak bounds, candidate symmetric binary
groups and bilinear terms (pass `llm=True` to add LLM suggestions if an LLM
is configured). `discopt install-skills` installs Claude Code commands
including `/reformulate`, which reviews a model and proposes changes. Both
only *suggest*: you apply a change, then confirm that the optimum is
unchanged and the solve got faster.

In [1]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [2]:
import numpy as np, discopt as do

# A small symmetric MILP: assign jobs to identical machines, minimise makespan.
def build(symbreak: bool, p=np.array([3, 5, 2, 4]), n_mach=3):
    m = do.Model("makespan_sb" if symbreak else "makespan")
    n_jobs = len(p)
    x = m.binary("x", shape=(n_jobs, n_mach))
    C = m.continuous("C", lb=0, ub=float(p.sum()))
    for j in range(n_jobs):
        m.subject_to(dm.sum(x[j, :]) == 1)
    for k in range(n_mach):
        m.subject_to(p @ x[:, k] <= C)
    if symbreak:
        # Bin-load lex: heaviest machine first.
        loads = [p @ x[:, k] for k in range(n_mach)]
        for k in range(n_mach - 1):
            m.subject_to(loads[k] >= loads[k + 1])
    m.minimize(C)
    return m, x, C

# milp_backend picks the engine for a pure MILP: "highs" (the default) or
# "native" (discopt's own MILP tree, which does no symmetry handling).
instances = {"4 jobs / 3 machines": dict(),
             "8 jobs / 3 machines": dict(p=np.array([2, 2, 3, 8, 7, 9, 6, 6]), n_mach=3),
             "8 jobs / 4 machines": dict(p=np.array([7, 5, 6, 3, 8, 4, 2, 9]), n_mach=4)}
for backend in ["highs", "native"]:
    for name, kw in instances.items():
        for label, sb in [("no symmetry breaking", False), ("bin-load lex", True)]:
            mm, _, _ = build(sb, **kw)
            r = mm.solve(milp_backend=backend)
            print(f"{backend:6s} {name:19s} {label:21s} nodes={r.node_count:4d}  C*={r.objective:.4g}")
# HiGHS handles this symmetry itself. On discopt's own tree the lex rows
# usually cut the node count, but not always (the last instance): they also
# change the LP relaxation and the branching order, so measure, don't assume.

highs  4 jobs / 3 machines no symmetry breaking  nodes=   1  C*=5
highs  4 jobs / 3 machines bin-load lex          nodes=   1  C*=5
highs  8 jobs / 3 machines no symmetry breaking  nodes=   1  C*=15
highs  8 jobs / 3 machines bin-load lex          nodes=   1  C*=15
highs  8 jobs / 4 machines no symmetry breaking  nodes=   1  C*=11
highs  8 jobs / 4 machines bin-load lex          nodes=   1  C*=11
native 4 jobs / 3 machines no symmetry breaking  nodes=  39  C*=5
native 4 jobs / 3 machines bin-load lex          nodes=   9  C*=5
native 8 jobs / 3 machines no symmetry breaking  nodes= 521  C*=15
native 8 jobs / 3 machines bin-load lex          nodes= 305  C*=15
native 8 jobs / 4 machines no symmetry breaking  nodes=  25  C*=11
native 8 jobs / 4 machines bin-load lex          nodes=  93  C*=11


## 5. Reformulation as a research practice

When stuck on a slow MILP/MINLP:

1. Look at the LP relaxation gap. If big, formulation is weak.
2. Inspect big-M values; tighten via OBBT.
3. Check for symmetry; add lex constraints.
4. Look for disjunctive structure; consider convex-hull reformulation.
5. Look for bilinear/concave terms; apply tighter relaxation.

This loop is the difference between a 2-day solve and a 2-second solve.

## References
{cite:p}`Margot2010,Liberti2009,Williams2013`.